# Fungal analysis

## Import data

In [1]:
!qiime tools import \
  --type 'SampleData[PairedEndSequencesWithQuality]' \
  --input-path data \
  --input-format CasavaOneEightSingleLanePerSampleDirFmt \
  --output-path raw_input.qza


!qiime demux summarize \
     --i-data raw_input.qza \
     --p-n 10000 \
     --verbose \
     --o-visualization raw_input.qzv

Imported data as CasavaOneEightSingleLanePerSampleDirFmt to raw_input.qza
/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_summarize/_visualizer.py:182: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  context['result_data'] = pd.concat([context['result_data'], df])
/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_summarize/_visualizer.py:76: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['213 nts' '234 nts' '260 nts' '274 nts' '300 nts' '301 nts' '301 nts']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  stats[stats.index != 'count'] = \
/Users/ac04

## Remove primers, only keep reads with primers

In [2]:
!qiime cutadapt trim-paired \
  --i-demultiplexed-sequences raw_input.qza \
  --p-cores 30 \
  --p-front-f CTTGGTCATTTAGAGGAAGTAA \
  --p-front-r GCTGCGTTCTTCATCGATGC \
  --p-discard-untrimmed \
  --o-trimmed-sequences trimmed_seqs.qza \
  --verbose > trim.log

Done           00:00:00        11,476 reads @  43.6 µs/read;   1.38 M reads/minute
Done           00:00:00         8,300 reads @  56.3 µs/read;   1.07 M reads/minute
Done           00:00:00        11,734 reads @  41.7 µs/read;   1.44 M reads/minute
Done           00:00:00        96,935 reads @   7.5 µs/read;   7.96 M reads/minute
Done           00:00:00        58,644 reads @  10.5 µs/read;   5.74 M reads/minute
Done           00:00:00       141,058 reads @   5.8 µs/read;  10.43 M reads/minute
Done           00:00:00       111,888 reads @   6.4 µs/read;   9.32 M reads/minute
Done           00:00:00       114,347 reads @   6.7 µs/read;   8.89 M reads/minute
Done           00:00:00        75,517 reads @   9.2 µs/read;   6.50 M reads/minute
Done           00:00:00       204,882 reads @   4.7 µs/read;  12.77 M reads/minute
Done           00:00:00        56,620 reads @  11.1 µs/read;   5.43 M reads/minute
Done           00:00:00       123,121 reads @   6.7 µs/read;   9.00 M reads/minute
Done

In [3]:
!qiime demux summarize \
     --i-data trimmed_seqs.qza \
     --p-n 10000 \
     --verbose \
     --o-visualization trimmed_seqs.qzv

/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_summarize/_visualizer.py:182: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  context['result_data'] = pd.concat([context['result_data'], df])
/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_summarize/_visualizer.py:76: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['191 nts' '213 nts' '238 nts' '251 nts' '278 nts' '279 nts' '279 nts']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  stats[stats.index != 'count'] = \
/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_s

## Denoise/deduplicate merge seqs

In [4]:
!qiime dada2 denoise-paired \
    --i-demultiplexed-seqs trimmed_seqs.qza \
    --p-trunc-len-f  250 \
    --p-trunc-len-r 220 \
    --p-n-threads 30 \
    --o-table 2020_dada_table.qza \
    --o-representative-sequences 2020_dada_sequences.qza \
    --o-denoising-stats 2020_dada_stats.qza \
    --o-base-transition-stats 2020_dada_transitions.qza \
    --verbose >> dada.log



Loading required package: Rcpp
/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/rachis/metadata/metadata.py:610: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype 

In [9]:
!qiime metadata tabulate \
    --m-input-file 2020_dada_stats.qza \
    --o-visualization 2020_dada_stats.qzv

!qiime feature-table summarize \
    --i-table 2020_dada_table.qza \
    --output-dir dadaRes \
    --m-metadata-file fungiMap_2020.txt

Saved Visualization to: 2020_dada_stats.qzv
Saved ImmutableMetadata to: dadaRes/feature_frequencies.qza
Saved ImmutableMetadata to: dadaRes/sample_frequencies.qza
Saved Visualization to: dadaRes/summary.qzv


## Filter

Remove the Clermontia samples as these were not present in the 2023 data and other samples.

In [10]:
!qiime feature-table filter-samples \
  --i-table 2020_dada_table.qza \
  --m-metadata-file fungiMap_2020.txt \
  --p-where "Plant!='Clermontia sp.' AND Preferred!='NA'" \
  --o-filtered-table 2020_dada_table_filter_samples.qza

!qiime feature-table summarize \
    --i-table 2020_dada_table_filter_samples.qza \
    --output-dir dadaRes2 \
    --m-metadata-file fungiMap_2020.txt

Saved FeatureTable[Frequency] to: 2020_dada_table_filter_samples.qza
Saved ImmutableMetadata to: dadaRes2/feature_frequencies.qza
Saved ImmutableMetadata to: dadaRes2/sample_frequencies.qza
Saved Visualization to: dadaRes2/summary.qzv


Remove ASVs that are present in a single sample or that have a total frequency lower than 0.1% of the average read depth of the samples (~35, average reads per sample = 32.502)

In [1]:
!qiime feature-table filter-features \
  --i-table 2020_dada_table_filter_samples.qza \
  --p-min-samples 2 \
  --p-min-frequency 35 \
  --o-filtered-table 2020_filtered_table.qza
!qiime feature-table summarize \
    --i-table 2020_filtered_table.qza \
    --output-dir dadaFiltered \
    --m-metadata-file fungiMap_2020.txt

Saved FeatureTable[Frequency] to: 2020_filtered_table.qza
Saved ImmutableMetadata to: dadaFiltered/feature_frequencies.qza
Saved ImmutableMetadata to: dadaFiltered/sample_frequencies.qza
Saved Visualization to: dadaFiltered/summary.qzv


In [2]:
!qiime feature-table filter-seqs \
  --i-data 2020_dada_sequences.qza \
  --i-table 2020_filtered_table.qza \
  --o-filtered-data 2020_filtered_sequences.qza


Saved FeatureData[Sequence] to: 2020_filtered_sequences.qza
